# ECG Arrhythmia Classification — Median Filter Experiment

This notebook adds the **median filter** as a lightweight ECG denoising method and compares it with the baseline ECG waveform input.

Experiment design:

- **P1 Baseline**: existing preprocessed ECG beats → z-normalization → classifier
- **P2 Median**: existing preprocessed ECG beats → z-normalization → short-window median filter → same classifier

The classifier is intentionally kept the same for both phases so the comparison isolates the effect of median filtering.

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
from __future__ import annotations

import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline

# Make imports work from either local project root, Kaggle, or VS Code/Jupyter.
def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for c in candidates:
        if (c / "src" / "utils" / "ecg_denoiser_utils.py").exists():
            return c
    # Fallback for Kaggle/manual use. Change this if your project is somewhere else.
    return Path("/kaggle/working/IT2022_Projects-main")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.ecg_denoiser_utils import (
    CLASSES_ORDER,
    build_dataset,
    beats_to_arrays,
    zscore_waveform,
    print_metrics,
    evaluate_predictions,
)
from src.model.ecg.median_denoiser import median_denoise_batch, median_feature_summary

OUT_DIR = PROJECT_ROOT / "outputs" / "ecg_median"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUT_DIR:", OUT_DIR)

In [ ]:
# =============================================================================
# DATA PATH
# =============================================================================
# The project zip already contains ECG csv/txt files. This resolver also supports Kaggle-style paths.
def resolve_ecg_dir() -> Path:
    candidates = [
        PROJECT_ROOT / "src" / "data" / "ecg_data" / "mitbih_database",
        PROJECT_ROOT / "src" / "data" / "ecg_data",
        Path("/kaggle/input/mitbih-database"),
        Path("/kaggle/input/mit-bih-arrhythmia-database"),
    ]
    for c in candidates:
        if c.exists() and any(c.glob("100.csv")):
            return c
    raise FileNotFoundError("Could not find MIT-BIH csv files. Update resolve_ecg_dir().")

ECG_DIR = resolve_ecg_dir()
print("ECG_DIR:", ECG_DIR)

In [ ]:
# =============================================================================
# LOAD BEATS
# =============================================================================
# This uses the existing project utility: baseline removal + bandpass + beat segmentation.
# Therefore, median filtering here is tested as an extra denoising step on the segmented beat tensor.

train_beats, test_beats = build_dataset(str(ECG_DIR))

W_tr, F_tr, RR_tr, y_tr_text, pid_tr = beats_to_arrays(train_beats)
W_te, F_te, RR_te, y_te_text, pid_te = beats_to_arrays(test_beats)

print("Train waveform:", W_tr.shape, "Train handcrafted:", F_tr.shape)
print("Test waveform :", W_te.shape, "Test handcrafted :", F_te.shape)
print("Train labels:", pd.Series(y_tr_text).value_counts().to_dict())
print("Test labels :", pd.Series(y_te_text).value_counts().to_dict())

In [ ]:
# =============================================================================
# LABEL ENCODING + BASELINE WAVEFORM NORMALIZATION
# =============================================================================
le = LabelEncoder()
le.fit(CLASSES_ORDER)

y_tr = le.transform(y_tr_text)
y_te = le.transform(y_te_text)
label_list = list(le.classes_)

# Existing baseline for segmented ECG beats: per-beat z-normalization.
X_w_tr_base = zscore_waveform(W_tr)
X_w_te_base = zscore_waveform(W_te)

print("Classes:", label_list)
print("Baseline waveform shape:", X_w_tr_base.shape)

In [ ]:
# =============================================================================
# APPLY MEDIAN FILTER
# =============================================================================
# For 180-sample ECG beats, use a small kernel.
# kernel_size=3 is safest; kernel_size=5 usually removes more spikes but may smooth small details.

MEDIAN_KERNEL = 5

X_w_tr_med = median_denoise_batch(X_w_tr_base, kernel_size=MEDIAN_KERNEL, show_progress=True)
X_w_te_med = median_denoise_batch(X_w_te_base, kernel_size=MEDIAN_KERNEL, show_progress=True)

print("Median waveform shape:", X_w_tr_med.shape)
print("Example summary:")
median_feature_summary(X_w_tr_base[0, :, 0], X_w_tr_med[0, :, 0], kernel_size=MEDIAN_KERNEL)

In [ ]:
# =============================================================================
# VISUAL CHECK
# =============================================================================
idx = 0
plt.figure(figsize=(10, 4))
plt.plot(X_w_tr_base[idx, :, 0], label="Baseline z-normalized beat", linewidth=1.5)
plt.plot(X_w_tr_med[idx, :, 0], label=f"Median filtered, k={MEDIAN_KERNEL}", linewidth=1.5)
plt.title("ECG beat before vs after median filtering")
plt.xlabel("Sample index")
plt.ylabel("Amplitude")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "ecg_median_example_beat.png", dpi=160)
plt.show()

In [ ]:
# =============================================================================
# FEATURE MATRICES FOR FAIR ABLATION
# =============================================================================
# We use the same handcrafted/RR features for both phases and only change the waveform stream.
# This isolates the effect of median filtering on the waveform input.

def make_ml_features(W, F):
    W_flat = W[:, :, 0].reshape(len(W), -1)
    return np.hstack([W_flat, F])

X_tr_base = make_ml_features(X_w_tr_base, F_tr)
X_te_base = make_ml_features(X_w_te_base, F_te)

X_tr_med = make_ml_features(X_w_tr_med, F_tr)
X_te_med = make_ml_features(X_w_te_med, F_te)

print("Baseline ML features:", X_tr_base.shape)
print("Median ML features  :", X_tr_med.shape)

In [ ]:
# =============================================================================
# TRAIN SAME CLASSIFIER ON BASELINE AND MEDIAN-FILTERED INPUTS
# =============================================================================
def train_eval(name, X_train, X_test):
    clf = make_pipeline(
        StandardScaler(),
        ExtraTreesClassifier(
            n_estimators=350,
            random_state=SEED,
            class_weight="balanced",
            n_jobs=-1,
        ),
    )
    clf.fit(X_train, y_tr)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)
    metrics = evaluate_predictions(y_te, y_pred, y_proba, n_cls=len(label_list))
    print_metrics(name, metrics)
    print(classification_report(y_te, y_pred, target_names=label_list, zero_division=0))
    return clf, y_pred, y_proba, metrics

base_clf, base_pred, base_proba, base_metrics = train_eval("P1_BASELINE", X_tr_base, X_te_base)
med_clf, med_pred, med_proba, med_metrics = train_eval("P2_MEDIAN", X_tr_med, X_te_med)

In [ ]:
# =============================================================================
# RESULT TABLE
# =============================================================================
results = pd.DataFrame([
    {"phase": "P1_BASELINE", **base_metrics},
    {"phase": f"P2_MEDIAN_k{MEDIAN_KERNEL}", **med_metrics},
])

results.to_csv(OUT_DIR / "ecg_median_results.csv", index=False)
results

In [ ]:
# =============================================================================
# CONFUSION MATRICES
# =============================================================================
def plot_cm(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(label_list)))
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(range(len(label_list)))
    ax.set_yticks(range(len(label_list)))
    ax.set_xticklabels(label_list, rotation=30)
    ax.set_yticklabels(label_list)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(OUT_DIR / filename, dpi=160)
    plt.show()

plot_cm(y_te, base_pred, "P1 Baseline confusion matrix", "cm_ecg_baseline.png")
plot_cm(y_te, med_pred, f"P2 Median k={MEDIAN_KERNEL} confusion matrix", "cm_ecg_median.png")

## How to connect this notebook to the existing LSTM backbone

The core integration is only this replacement:

```python
X_w_tr_base = zscore_waveform(W_tr)
X_w_te_base = zscore_waveform(W_te)

X_w_tr_med = median_denoise_batch(X_w_tr_base, kernel_size=5)
X_w_te_med = median_denoise_batch(X_w_te_base, kernel_size=5)

# Then pass X_w_tr_med / X_w_te_med into the same classifier instead of X_w_tr_base / X_w_te_base.
```

For the report, describe this as an ablation: the classifier, labels, train/test split, RR features, and handcrafted features are kept unchanged; only the waveform input is median-filtered.